In [ ]:
import torch
import matplotlib.pyplot as plt

In [ ]:
class IsotropicGaussianMixture:

    def __init__(self, weights, means, variances):
        self.weights = torch.tensor(weights)
        self.means = torch.tensor(means)
        self.variances = torch.tensor(variances)
        self.data_dim = len(weights)

    def density(self, x):
        # x: [batch_size, data_dim].
        p = 0
        for w, mu, sigma2 in zip(self.weights, self.means, self.variances):
            normalization = (2 * torch.pi * sigma2) ** (self.data_dim / 2)
            exponent = -1 / (2 * sigma2) * (x - mu).norm(dim=-1) ** 2
            p += w * torch.exp(exponent) / normalization
        return p

    def score_fn(self, x):
        # x: [batch_size, data_dim].
        s = torch.zeros_like(x)
        for w, mu, sigma2 in zip(self.weights, self.means, self.variances):
            cte = (2 * torch.pi * sigma2) ** (- self.data_dim / 2) / sigma2
            exponent = -1 / (2 * sigma2) * (x - mu).norm(dim=-1) ** 2
            s += w * cte * torch.exp(exponent).unsqueeze(dim=-1) * (x - mu)
        return s / -self.density(x).unsqueeze(dim=-1)

In [ ]:
class LangevinDynamics:

    def __init__(self, distribution):
        self.distribution = distribution
        self.data_dim = len(distribution.means[0])

    def langevin_sampling(self, n_samples, step_size, n_steps):
        x = torch.rand((n_samples, self.data_dim))
        trajectories = [x]

        with torch.no_grad():
            for _ in range(n_steps):
                epsilon = torch.randn_like(x)
                score = self.distribution.score_fn(x)
                x = x + step_size * score + (2 * step_size) ** 0.5 * epsilon
                trajectories.append(x)

        return torch.stack(trajectories, dim=1)

    def plot_trajectories(self, n_samples, step_size=5e-3, n_steps=1000):

        plt.figure(figsize=(8, 8))

        # Límites del gráfico:
        means = self.distribution.means
        variances = self.distribution.variances
        stds = variances.sqrt().unsqueeze(1).expand_as(means)
        max_x = torch.max(means[:, 0] + 3 * stds[:, 0]).item()
        min_x = torch.min(means[:, 0] - 3 * stds[:, 0]).item()
        max_y = torch.max(means[:, 1] + 3 * stds[:, 1]).item()
        min_y = torch.min(means[:, 1] - 3 * stds[:, 1]).item()

        # Mapa de calor para la densidad:
        x = torch.linspace(min_x, max_x, 100)
        y = torch.linspace(min_y, max_y, 100)
        X, Y = torch.meshgrid(x, y, indexing='xy')
        grid = torch.stack([X, Y], dim=-1).reshape(-1, self.data_dim)
        Z = self.distribution.density(grid).reshape(100, 100)
        plt.contourf(X, Y, Z, levels=100, cmap='viridis', zorder=0)

        # Campo vectorial del score:
        xq = torch.linspace(min_x, max_x, 30)
        yq = torch.linspace(min_y, max_y, 30)
        Xq, Yq = torch.meshgrid(xq, yq, indexing='xy')
        grid_q = torch.stack([Xq, Yq], dim=-1).reshape(-1, self.data_dim)
        scores = self.distribution.score_fn(grid_q)
        U = scores[:, 0].reshape(30, 30)
        V = scores[:, 1].reshape(30, 30)
        plt.quiver(Xq, Yq, U, V, color='white', alpha=0.2, width=0.001, zorder=1)

        # Trayectorias:
        trajectories = self.langevin_sampling(n_samples, step_size, n_steps)
        for traj in trajectories:
            plt.plot(traj[:, 0], traj[:, 1], linewidth=1, zorder=2)
            plt.scatter(traj[-1, 0], traj[-1, 1], s=30, color='black', zorder=3)

        plt.xlim(min_x, max_x)
        plt.ylim(min_y, max_y)
        plt.show()

In [ ]:
weights = [1/3, 1/3, 1/3]
#weights = [0.1, 0.7, 0.2]
means = [[6, 2], [-3, 5], [0, -5]]
variances = [1, 1, 1]
gmm = IsotropicGaussianMixture(weights, means, variances)

langevin = LangevinDynamics(gmm)
langevin.plot_trajectories(n_samples=5)